Classification: finetune
========================

In this notebook we illustrate how to re-train the models on user's data. Specifically, we remap the last layer of the model to the desired classes, without modifying the model's internal weights; this operation is called finetuning and is not as computationally intensive as re-training the full model. 
Regardless, this module greatly benefits from GPU compute, as long as the GPU(s) support CUDA and `nvidia-smi` is configured correctly. 

In [1]:
import argparse
import shutil
import sys
import os
from pathlib import Path

import numpy as np
import pandas as pd
import yaml

from mzbsuite.utils import cfg_to_arguments

We need to declare the running parameters for the script, 

In [2]:
from pathlib import Path
import yaml

cwd = Path.cwd()
ROOT_DIR = cwd.parent.absolute()

# Alternatively, change to your working directory:
# ROOT_DIR = Path("path/to/your/working/dir/here")

MODEL_C="convnext-small-v0"

arguments = {
    "input_dir": ROOT_DIR / "data/mzb_example_data/training_dataset", 
    "save_model": ROOT_DIR / f"models/mzb-classification-models/{MODEL_C}/checkpoints", 
    "config_file": ROOT_DIR / "configs/mzb_example_config.yaml"
}

with open(str(arguments["config_file"]), "r") as f:
    cfg = yaml.load(f, Loader=yaml.FullLoader)

Convert these parameters to a dictionary: 

In [3]:
from mzbsuite.utils import cfg_to_arguments

# Transforms configurations dicts to argparse arguments
args = cfg_to_arguments(arguments)
cfg = cfg_to_arguments(cfg)
print(str(cfg))

{'glob_random_seed': 222, 'glob_root_folder': '/home/mzbuser/work/mzb-suite', 'glob_blobs_folder': '/home/mzbuser/work/mzb-suite/data/derived/blobs/', 'glob_local_format': 'pdf', 'model_logger': 'wandb', 'impa_image_format': 'jpg', 'impa_clip_areas': [2700, 4700, -1, -1], 'impa_area_threshold': 5000, 'impa_gaussian_blur': [21, 21], 'impa_gaussian_blur_passes': 3, 'impa_adaptive_threshold_block_size': 351, 'impa_mask_postprocess_kernel': [11, 11], 'impa_mask_postprocess_passes': 5, 'impa_bounding_box_buffer': 200, 'impa_save_clips_plus_features': True, 'lset_class_cut': 'order', 'lset_val_size': 0.1, 'trcl_learning_rate': 0.0001, 'trcl_batch_size': 8, 'trcl_weight_decay': 0, 'trcl_step_size_decay': 5, 'trcl_number_epochs': 75, 'trcl_save_topk': 1, 'trcl_num_classes': 8, 'trcl_model_pretrarch': 'convnext-small', 'trcl_num_workers': 16, 'trcl_wandb_project_name': 'mzb-classifiers', 'trcl_logger': 'wandb', 'trsk_learning_rate': 0.001, 'trsk_batch_size': 32, 'trsk_weight_decay': 0, 'trsk_st

Before starting the training, we initialise a utility for tracking experiments(i.e. how the training is progressing, with accuracy metrics etc). 

We support logging training progress with either [Weights & Biases](https://wandb.ai/) or [Tensorflow](https://www.tensorflow.org/). 

Here use Weights & Biases.

In [12]:
import wandb

wandb.init()

wandb: ERROR The nbformat package was not found. It is required to save notebook history.


The tracking is setup as anonymous user here; if you want to track your experiments using their online dashboard, you will need to create an account and link it to your session with an API key. 

Please consult Weights & Biases or Tensorflow documentation for that. 

With everything set up, we move on to the model finetuning, using the script `classification/main_classification_finetune.py`. 

In [4]:
from scripts.classification.main_classification_finetune import main as classification

classification(args, cfg)

Loading model from d:\mzb-suite\models\mzb-classification-models\convnext-small-v0\checkpoints


wandb: WARNING The anonymous setting has no effect and will be removed in a future version.
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\Pegoraro\_netrc.
wandb: Currently logged in as: lpego (biodetect) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: logging graph, to disable use `wandb.watch(log_graph=False)`
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
c:\Users\Pegoraro\AppData\Local\miniforge3\envs\mzbsuite\lib\site-packages\pytorch_lightning\callbacks\model_checkpoint.py:881: Checkpoint directory D:\mzb-suite\models\mzb-classification-models\convnext-small-v0\checkpoints exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name     ┃ Type              ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model    │ ConvNeXt          │ 49.5 M │ train │     0 │
│ 1 │ accuracy │ MulticlassF1Score │      0 │ train │     0 │
└───┴──────────┴───────────────────┴────────┴───────┴───────┘

Trainable params: 6.2 K                                                                                            
Non-trainable params: 49.5 M                                                                                       
Total params: 49.5 M                                                                                               
Total estimated model params size (MB): 197                                                                        
Modules in train mode: 384                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

...

c:\Users\Pegoraro\AppData\Local\miniforge3\envs\mzbsuite\lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:429: Consider setting `persistent_workers=True` in 'train_dataloader' to speed up the dataloader worker initialization.


Epoch 0: 100%|██████████| 28/28 [00:48<00:00,  0.57it/s, v_num=vpwc, trn_loss=0.233]  


Detected KeyboardInterrupt, attempting graceful shutdown ...


SystemExit: 1

c:\Users\Pegoraro\AppData\Local\miniforge3\envs\mzbsuite\lib\site-packages\IPython\core\interactiveshell.py:3587: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
